In [1]:
###### Hello World test ######
from PyQt6.QtWidgets import QApplication, QWidget

# Only needed for access to command line arguments
import sys

# You need one (and only one) QApplication instance per application.
# Pass in sys.argv to allow command line arguments for your app.
# If you know you won't use command line arguments QApplication([]) works too.
app = QApplication(sys.argv)

# Create a Qt widget, which will be our window.
window = QWidget()
window.show()  # IMPORTANT!!!!! Windows are hidden by default.

# Start the event loop.
app.exec()


# Your application won't reach here until you exit and the event
# loop has stopped.



0

In [2]:
###### Basic Window ######
import sys
from PyQt6.QtWidgets import QApplication, QMainWindow, QVBoxLayout, QWidget, QPushButton, QSlider, QLabel, QLineEdit, QHBoxLayout
from PyQt6.QtCore import Qt
import vtkmodules.all as vtk
from vtk.qt.QVTKRenderWindowInteractor import QVTKRenderWindowInteractor

class MainWindow(QMainWindow):
    def __init__(self, parent=None):
        super(MainWindow, self).__init__(parent)
        
        self.setWindowTitle("VTK with PyQt6")

        self.frame = QWidget()
        self.layout = QVBoxLayout()

        # Set the window size to be 800x600 pixels and position it at 100, 100
        self.setGeometry(100, 100, 800, 600)
        
        # Initializing the VTK Render Widget into the Qt layout
        self.vtk_widget = QVTKRenderWindowInteractor(self.frame)
        self.layout.addWidget(self.vtk_widget)
        
        # Controls
        self.controls_layout = QHBoxLayout()
        # Slider's label
        self.slider_label = QLabel("Area Percent Change:")
        self.controls_layout.addWidget(self.slider_label)
        # Slider for aneurysm area increase
        self.area_slider = QSlider(Qt.Orientation.Horizontal)
        self.area_slider.setRange(100, 1000)
        self.area_slider.setValue(500)
        self.controls_layout.addWidget(self.area_slider)
        # Button for running the deformation
        self.run_button = QPushButton("Run Deformation")
        self.controls_layout.addWidget(self.run_button)
        # Add controls layout to main layout
        self.layout.addLayout(self.controls_layout)
        self.frame.setLayout(self.layout)
        self.setCentralWidget(self.frame)
        # Connect the button to the run_deformation method
        self.run_button.clicked.connect(self.run_deformation)
        
        # VTK Setup
        self.vtk_interactor = self.vtk_widget.GetRenderWindow().GetInteractor()
        self.ren = vtk.vtkRenderer()
        self.vtk_widget.GetRenderWindow().AddRenderer(self.ren)
        
        self.initialize_vtk()

    def initialize_vtk(self):
        # Load initial VTK files and set up the scene here
        # Example:
        # self.mesh = load_vtp_file("mesh-complete-exterior.vtp")
        # self.centerline = load_vtp_file("centerline.vtp")
        # self.ren.AddActor(mesh_actor)
        # self.ren.AddActor(centerline_actor)
        # self.ren.ResetCamera()
        pass

    def run_deformation(self):
        area_percent_change = self.area_slider.value()
        print(f"Running deformation with area percent change: {area_percent_change}")
        # Trigger the deformation logic here
        # Example:
        # create_aneurysm(self.mesh, self.centerline, self.selected_points, area_percent_change)
        # self.update_vtk_view()
    
    def update_vtk_view(self):
        # Update VTK view after deformation
        self.vtk_widget.GetRenderWindow().Render()

if __name__ == "__main__":
    app = QApplication(sys.argv)
    window = MainWindow()
    window.show()
    sys.exit(app.exec())


: 

In [1]:
###### Full VTK Integration ######
import sys
from PyQt6.QtWidgets import QApplication, QMainWindow, QVBoxLayout, QWidget, QPushButton, QSlider, QLabel, QHBoxLayout, QFileDialog, QLineEdit
from PyQt6.QtCore import Qt
import vtkmodules.all as vtk
from vtk.qt.QVTKRenderWindowInteractor import QVTKRenderWindowInteractor
from vtk_module import VTKHandler

class MainWindow(QMainWindow):
    def __init__(self, parent=None):
        super(MainWindow, self).__init__(parent)
        
        self.setWindowTitle("VTK with PyQt6")
        
        self.frame = QWidget()
        self.layout = QVBoxLayout()
        
        # Set the window size to be 800x600 pixels and position it at 100, 100
        self.setGeometry(100, 100, 1200, 900)
        
        # VTK Render Widget
        self.vtk_widget = QVTKRenderWindowInteractor(self.frame)
        self.layout.addWidget(self.vtk_widget)
        
        # Controls
        self.controls_layout = QHBoxLayout()

        # Import Buttons
        self.import_mesh_button = QPushButton("Import Mesh")
        self.import_mesh_button.clicked.connect(self.import_mesh)
        self.controls_layout.addWidget(self.import_mesh_button)
        self.import_centerline_button = QPushButton("Import Centerline")
        self.import_centerline_button.clicked.connect(self.import_centerline)
        self.controls_layout.addWidget(self.import_centerline_button)

        # Slider's label
        self.slider_label = QLabel("Area Percent Change:")
        self.controls_layout.addWidget(self.slider_label)
        # Slider for aneurysm area increase
        self.area_slider = QSlider(Qt.Orientation.Horizontal)
        self.area_slider.setRange(100, 1000)
        self.area_slider.setValue(500)
        self.controls_layout.addWidget(self.area_slider)
        # Display the slider value
        self.slider_value = QLineEdit()
        self.slider_value.setText(f"{self.area_slider.value()}%")
        self.slider_value.setFixedWidth(50)
        self.controls_layout.addWidget(self.slider_value)
        self.area_slider.valueChanged.connect(lambda value: self.slider_value.setText(f"{value}%"))
        self.slider_value.textChanged.connect(lambda text: self.area_slider.setValue(int(text.replace("%", ""))))
        # Button for showing selectable nodes on the centerline
        self.show_nodes_button = QPushButton("Select Nodes")
        self.controls_layout.addWidget(self.show_nodes_button)
        # Button for running the deformation
        self.run_button = QPushButton("Create Aneurysm")
        self.controls_layout.addWidget(self.run_button)
        # Add controls layout to main layout
        self.layout.addLayout(self.controls_layout)
        self.frame.setLayout(self.layout)
        self.setCentralWidget(self.frame)
        # Connect the button to the run_deformation method
        self.run_button.clicked.connect(self.run_deformation)
        # Connect the button to the show_nodes method
        self.show_nodes_button.clicked.connect(self.display_centerline_nodes)
        # To add a second row of buttons, add another QHBoxLayout and add it to the main layout
        self.controls_layout2 = QHBoxLayout()
        self.button2 = QPushButton("Button 2")
        self.controls_layout2.addWidget(self.button2)
        self.layout.addLayout(self.controls_layout2)
        
        # VTK Setup
        self.vtk_interactor = self.vtk_widget.GetRenderWindow().GetInteractor()
        self.vtk_handler = None
        # VTKHandler("input/mesh-complete-exterior.vtp", "input/centerline.vtp")

    def import_mesh(self):
        file_name, _ = QFileDialog.getOpenFileName(self, "Import Mesh", "", "VTK Files (*.vtp)")
        if file_name:
            self.mesh_file = file_name
            if hasattr(self, 'centerline_file'):
                self.initialize_vtk_handler()

    def import_centerline(self):
        file_name, _ = QFileDialog.getOpenFileName(self, "Import Centerline", "", "VTK Files (*.vtp)")
        if file_name:
            self.centerline_file = file_name
            if hasattr(self, 'mesh_file'):
                self.initialize_vtk_handler()

    def initialize_vtk_handler(self):
        self.vtk_handler = VTKHandler(self.mesh_file, self.centerline_file)
        self.ren = self.vtk_handler.get_renderer()
        self.vtk_widget.GetRenderWindow().AddRenderer(self.ren)

        self.style = self.vtk_handler.get_interactor_style(self.vtk_interactor)
        self.vtk_interactor.SetInteractorStyle(self.style)

        # self.vtk_widget.GetRenderWindow().Render()

        self.vtk_interactor.Initialize()
        self.vtk_interactor.Start()

    def run_deformation(self):
        area_percent_change = self.area_slider.value()
        print(f"Running deformation with area percent change: {area_percent_change}")
        self.style.deform_mesh(area_percent_change)
        # self.vtk_handler.get_interactor_style(self.vtk_interactor).deform_mesh()
        # self.vtk_widget.GetRenderWindow().Render()
    
    def display_centerline_nodes(self):
        print("Please select three centerline nodes to generate aneurysm.")
        self.style.display_vertices()
    
if __name__ == "__main__":
    app = QApplication(sys.argv)
    window = MainWindow()
    window.show()
    sys.exit(app.exec())


ValueError: invalid literal for int() with base 10: ''

ValueError: invalid literal for int() with base 10: ''

ValueError: invalid literal for int() with base 10: '778.'

ValueError: invalid literal for int() with base 10: '778.2'

Please select three centerline nodes to generate aneurysm.
Picked actor centerpointID: 348
Picked actor centerpointID: 361
Picked actor centerpointID: 373
Running deformation with area percent change: 331


2024-06-04 11:34:51.062 (  70.544s) [    7F8AE2776280]vtkPolyDataPlaneCutter.:589   INFO| Executing vtkPolyData plane cutter
2024-06-04 11:34:51.238 (  70.720s) [    7F8AE2776280]vtkPolyDataPlaneCutter.:589   INFO| Executing vtkPolyData plane cutter
2024-06-04 11:34:51.259 (  70.741s) [    7F8AE2776280]vtkPolyDataPlaneCutter.:589   INFO| Executing vtkPolyData plane cutter
2024-06-04 11:34:51.276 (  70.758s) [    7F8AE2776280]vtkPolyDataPlaneCutter.:589   INFO| Executing vtkPolyData plane cutter
2024-06-04 11:34:51.446 (  70.928s) [    7F8AE2776280]vtkPolyDataPlaneCutter.:589   INFO| Executing vtkPolyData plane cutter
2024-06-04 11:34:51.467 (  70.949s) [    7F8AE2776280]vtkPolyDataPlaneCutter.:589   INFO| Executing vtkPolyData plane cutter
2024-06-04 11:34:51.483 (  70.965s) [    7F8AE2776280]vtkPolyDataPlaneCutter.:589   INFO| Executing vtkPolyData plane cutter
2024-06-04 11:34:51.649 (  71.131s) [    7F8AE2776280]vtkPolyDataPlaneCutter.:589   INFO| Executing vtkPolyData plane cutter
